# Notebook 02 — Cohort Retention Analysis
**Tujuan:** Hitung retention rate per cohort bulanan dan visualisasikan sebagai heatmap.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

OUT     = Path('../output')
FIGURES = OUT / 'figures'
FIGURES.mkdir(exist_ok=True)

BLUE = '#2563EB'
GRAY = '#6B7280'
sns.set_theme(style='whitegrid', font_scale=1.0)

df_orders = pd.read_parquet(OUT / 'df_orders.parquet')
df_orders['order_month'] = df_orders['order_month'].astype('period[M]')
print(f'Loaded: {len(df_orders):,} orders, {df_orders["customer_id"].nunique():,} customers')

## 1. Assign Cohort Month

In [ ]:
# First purchase month per customer
first_purchase = (
    df_orders.groupby('customer_id')['order_date']
    .min()
    .dt.to_period('M')
    .rename('cohort_month')
)

df_orders = df_orders.join(first_purchase, on='customer_id')

# Cohort index: selisih bulan antara order_month dan cohort_month
df_orders['cohort_index'] = (
    df_orders['order_month'] - df_orders['cohort_month']
).apply(lambda x: x.n)

print('Cohort index range:', df_orders['cohort_index'].min(),
      '→', df_orders['cohort_index'].max())
df_orders[['customer_id','order_month','cohort_month','cohort_index']].head(5)

## 2. Build Cohort Pivot Table

In [ ]:
cohort_data = (
    df_orders
    .groupby(['cohort_month', 'cohort_index'])['customer_id']
    .nunique()
    .reset_index()
)

cohort_pivot = cohort_data.pivot_table(
    index='cohort_month',
    columns='cohort_index',
    values='customer_id'
)

# Retention rate: bagi dengan cohort size (kolom 0)
cohort_size = cohort_pivot.iloc[:, 0]
retention   = cohort_pivot.divide(cohort_size, axis=0)

# Batasi ke 12 bulan pertama agar heatmap tidak terlalu lebar
retention = retention.iloc[:, :13]

print(f'Cohort pivot: {retention.shape[0]} cohorts × {retention.shape[1]} bulan')
print(f'\nAvg Month-1 Retention: {retention.iloc[:, 1].mean():.1%}')

## 3. Heatmap Cohort Retention

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))

# Format index sebagai string untuk label
retention_plot = retention.copy()
retention_plot.index = retention_plot.index.astype(str)
retention_plot.columns = [f'M+{c}' for c in retention_plot.columns]

sns.heatmap(
    retention_plot,
    annot=True,
    fmt='.0%',
    cmap='Blues',
    vmin=0, vmax=1,
    linewidths=0.5,
    linecolor='#E2E8F0',
    ax=ax,
    annot_kws={'size': 8},
    cbar_kws={'label': 'Retention Rate', 'shrink': 0.8}
)

ax.set_title('Customer Cohort Retention Rate — Olist (2017–2018)',
             fontweight='bold', fontsize=14, pad=15)
ax.set_xlabel('Bulan Setelah Akuisisi', fontsize=11)
ax.set_ylabel('Cohort (Bulan Pertama Beli)', fontsize=11)
ax.tick_params(axis='x', labelsize=9)
ax.tick_params(axis='y', labelsize=9, rotation=0)

# ── Callout best / worst M+1 cohort ─────────────────────────────────────
m1_vals = retention.iloc[:, 1]
best_cohort  = m1_vals.idxmax()
worst_cohort = m1_vals[m1_vals > 0].idxmin()
idx_list     = list(retention_plot.index)
best_row  = idx_list.index(str(best_cohort))  + 0.5
worst_row = idx_list.index(str(worst_cohort)) + 0.5
n_cols = retention_plot.shape[1]

ax.annotate(
    f'Best cohort\nM+1: {m1_vals[best_cohort]:.0%}',
    xy=(n_cols - 0.1, best_row), xytext=(n_cols + 1.5, best_row - 0.8),
    fontsize=8.5, color='#1D4ED8', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#1D4ED8', lw=1.5),
    va='center', ha='left', annotation_clip=False
)
ax.annotate(
    f'Worst cohort\nM+1: {m1_vals[worst_cohort]:.0%}',
    xy=(n_cols - 0.1, worst_row), xytext=(n_cols + 1.5, worst_row + 0.8),
    fontsize=8.5, color='#DC2626', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#DC2626', lw=1.5),
    va='center', ha='left', annotation_clip=False
)

# Industry benchmark note
fig.text(
    0.5, -0.01,
    'Benchmark industri e-commerce: M+1 retention 25–35% '
    '(Tokopedia/Shopee) — dataset ini jauh di bawah, peluang improvement besar',
    ha='center', fontsize=8.5, color='#6B7280', style='italic'
)

plt.tight_layout()
plt.savefig(FIGURES / 'A_cohort_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Cohort Size — Akuisisi Pelanggan Baru per Bulan

In [ ]:
cohort_size_plot = cohort_size.copy()
cohort_size_plot.index = cohort_size_plot.index.astype(str)
# Buang bulan pertama & terakhir (data tidak lengkap)
cohort_size_plot = cohort_size_plot.iloc[1:-1]

fig, ax = plt.subplots(figsize=(14, 4))
colors = [BLUE if v == cohort_size_plot.max() else '#93C5FD'
          for v in cohort_size_plot.values]
ax.bar(cohort_size_plot.index, cohort_size_plot.values, color=colors)
ax.set_xlabel('Cohort Bulan')
ax.set_ylabel('Pelanggan Baru')
ax.set_title('Akuisisi Pelanggan Baru per Cohort Bulan', fontweight='bold')
ax.tick_params(axis='x', rotation=45)
for i, v in enumerate(cohort_size_plot.values):
    ax.text(i, v + 30, f'{v:,}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES / 'B_cohort_size.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: B_cohort_size.png')

## 5. Simpan Cohort Retention

In [ ]:
retention.to_parquet(OUT / 'cohort_retention.parquet')
print('Tersimpan: cohort_retention.parquet')
print(f'Validasi M+0 = 100%: {(retention.iloc[:, 0] == 1.0).all()}')